In [ ]:
import os, time, glob, subprocess, shutil
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTModel

DEBUG = True
SR = 16000
SHARD_SIZE = 1000
BATCH = 32
NUM_WORKERS = 4
OUT_DIR = "/kaggle/working/ast_feats" + ("_debug" if DEBUG else "")
os.makedirs(OUT_DIR, exist_ok=True)

MASTER = "/kaggle/input/datasets/maheshkarrevula/fakeavceleb-master/master.csv"
df = pd.read_csv(MASTER)
if DEBUG:
    df = df.sample(50, random_state=0).reset_index(drop=True)

assert os.path.exists(df["path"].iloc[0]), "FakeAVCeleb dataset is not attached"
assert shutil.which("ffmpeg"), "ffmpeg not found"
print("videos:", len(df))
print("existing shards:", len(glob.glob(f"{OUT_DIR}/shard_*.npy")))   # should be 0 on a fresh run

In [ ]:
fe = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

def load_audio(path):
    cmd = ["ffmpeg", "-v", "quiet", "-i", path, "-vn", "-ac", "1", "-ar", str(SR),
           "-t", "10.24", "-f", "f32le", "-"]
    try:
        raw = subprocess.run(cmd, capture_output=True, timeout=60).stdout
        wav = np.frombuffer(raw, dtype=np.float32)
    except Exception:
        wav = np.zeros(0, dtype=np.float32)
    failed = len(wav) < SR // 2                        # less than 0.5 s of audio
    if failed:
        wav = np.zeros(SR, dtype=np.float32)
    return wav, failed

class AudioDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        wav, failed = load_audio(self.paths[i])
        x = fe(wav, sampling_rate=SR, return_tensors="pt")["input_values"][0]   # [1024,128]
        return x, failed

In [ ]:
device = "cuda"
ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593").to(device).half().eval()

@torch.no_grad()
def embed(x):
    return ast(input_values=x.to(device).half()).pooler_output.float().cpu().numpy().astype(np.float16)  # [B,768]

In [ ]:
n_shards = (len(df) + SHARD_SIZE - 1) // SHARD_SIZE
t0, done, n_failed = time.time(), 0, 0

for s in range(n_shards):
    out_f, fail_f = f"{OUT_DIR}/shard_{s:04d}.npy", f"{OUT_DIR}/fail_{s:04d}.npy"
    if os.path.exists(out_f) and os.path.exists(fail_f):
        print("skip", s); continue
    sub = df.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
    dl = DataLoader(AudioDS(sub["path"].tolist()), batch_size=BATCH,
                    num_workers=NUM_WORKERS, shuffle=False)
    feats, fails = [], []
    for x, failed in dl:
        feats.append(embed(x)); fails.append(failed.numpy())
    feats, fails = np.concatenate(feats), np.concatenate(fails)
    np.save(out_f, feats); np.save(fail_f, fails)
    done += len(sub); n_failed += int(fails.sum())
    print(f"shard {s+1}/{n_shards} | {feats.shape} | {(time.time()-t0)/done:.2f}s/video | failed so far: {n_failed}")

In [ ]:
if not DEBUG:
    feats = np.concatenate([np.load(f) for f in sorted(glob.glob(f"{OUT_DIR}/shard_*.npy"))])
    fails = np.concatenate([np.load(f) for f in sorted(glob.glob(f"{OUT_DIR}/fail_*.npy"))])
    assert len(feats) == len(df)
    np.save(f"{OUT_DIR}/ast_feats_all.npy", feats)
    np.save(f"{OUT_DIR}/ast_fail_all.npy", fails)
    print(feats.shape, feats.dtype, "failed videos:", fails.sum())